In [15]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable
from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number, to_timestamp, lit, create_map, upper, trim
from pyspark.sql.types import DecimalType
from helpers import get_table, get_bronze
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from functools import reduce
from itertools import chain
print(pyspark.__version__)

3.5.8


In [2]:
builder = (
    SparkSession.builder
    .appName("user_bronze")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

spark = configure_spark_with_delta_pip(
    builder,
    extra_packages=["org.postgresql:postgresql:42.7.3"]
).getOrCreate()

26/04/27 15:18:36 WARN Utils: Your hostname, CVPThuyTTD11-1 resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/27 15:18:36 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/thuyttd11/.local/lib/python3.8/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/thuyttd11/.ivy2/cache
The jars for the packages stored in: /home/thuyttd11/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-6c7e2084-60d8-4d5d-8d13-72e164596706;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.3.2 in central
	found io.delta#delta-storage;3.3.2 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.postgresql#postgresql;42.7.3 in central
	found org.checkerframework#checker-qual;3.42.0 in central
:: resolution report :: resolve 376ms :: artifacts dl 15ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.3.2 from central in [default]
	io.delta#delta-storage;3.3.2 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	org.checkerframework#checker-qual;3.42.0 from central in [default]
	org.postgresql#postgresql;42.7.3 from central in [default]
	------------------------

# payments
+ Standardize column names
+ Ensure dtypes: Cast data types for payment_id, subscription_id, amount, convert to timestamp and standardize timezone for payment_date
+ Derive date parts from payment_date: date, year, month
+ Remove or quarantine null/invalid records for payment_id, subscription_id
+ Normalize and validate categorical values for payment_status, currency, payment_method
+ Enforce grain and deduplicate by payment_id
+ Validate referential integrity for subscription_id
+ Derive indicator/flag columns for is_successful_payment, is_failed_payment, is_refunded_payment
+ Optionally derive reporting amount for amount_usd or other converted reporting currency
+ Store output as Delta table silver_payments_clean
+ Optimize table for downstream Gold usage

## Read Bronze payments data

In [3]:
# Bronze layer
payments_bronze = "/home/thuyttd11/subscription-analytics/spark-data/bronze/payments"
df_bronze_payments = get_bronze(payments_bronze, spark=spark)
df_bronze_payments.show(5)

26/04/27 15:18:48 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

+---+----------+---------------+------+--------+--------------+------------+--------------+--------------------+--------------------+--------------------+
| id|payment_id|subscription_id|amount|currency|payment_status|payment_date|payment_method|         ingest_time|   source_identifier|            batch_id|
+---+----------+---------------+------+--------+--------------+------------+--------------+--------------------+--------------------+--------------------+
|  1|         1|         177985| 10.40|     USD|       success|  2024-06-05|        paypal|2026-04-22 11:16:...|jdbc:postgresql:/...|app-2026042204161...|
|  2|         2|          33628| 13.98|     USD|       success|  2024-05-29|   credit_card|2026-04-22 11:16:...|jdbc:postgresql:/...|app-2026042204161...|
|  3|         3|           5872| 27.68|     USD|       success|  2024-11-03|   credit_card|2026-04-22 11:16:...|jdbc:postgresql:/...|app-2026042204161...|
|  4|         4|           3456| 12.46|     VND|       success|  2026-

## Remove duplicates based on the correct business grain

In [5]:
df1 = df_bronze_payments.drop("id", "source_identifier", "batch_id")
df1.columns

['payment_id',
 'subscription_id',
 'amount',
 'currency',
 'payment_status',
 'payment_date',
 'payment_method',
 'ingest_time']

## Remove duplicates for payment_id + payment_date
(should have minutes + second) based on business gain 

In [6]:
cols = df1.columns

# Ensure timestamps
df_payments_clean = (
    df1
    .select(*cols)
    .withColumn("payment_date", F.to_timestamp("payment_date"))
    .withColumn("ingest_time", F.to_timestamp("ingest_time"))
)

# Business grain: payment_id + payment_date
window_spec = (
    Window
    .partitionBy("payment_id", "payment_date")
    .orderBy(F.col("ingest_time").desc())
)

df_with_rn = df_payments_clean.withColumn(
    "rn", F.row_number().over(window_spec)
)

# Valid (deduplicated)
df2 = (
    df_with_rn
    .filter(F.col("rn") == 1)
    .drop("rn")
)

# Quarantine (duplicates)
df2_quarantine = (
    df_with_rn
    .filter(F.col("rn") > 1)
    .drop("rn")
)

# Checks
print("Valid count:", df2.count())
print("Quarantine count:", df2_quarantine.count())

Valid count: 350000


[Stage 22:==============>                                           (1 + 3) / 4]

Quarantine count: 350000


## Ensure dtypes
Cast data types for payment_id, subscription_id, amount, convert to timestamp and standardize timezone for payment_date

In [8]:
df3 = df2.select(
    col("payment_id").cast("bigint").alias("payment_id"),
    col("subscription_id").cast("bigint").alias("subscription_id"),
    col("amount").cast("decimal(10,2)").alias("amount"),
    col("currency").cast("string").alias("currency"),
    col("payment_status").cast("string").alias("payment_status"),
    to_timestamp(col("payment_date")).alias("payment_date"),
    col("payment_method").cast("string").alias("payment_method"),
    to_timestamp(col("ingest_time")).alias("ingest_time")
)
# df_silver_payments3.dtypes

In [88]:
df_silver_payments3.show(2)

[Stage 386:======================================>                  (2 + 1) / 3]

+----------+---------------+------+--------+--------------+-------------------+--------------+--------------------+
|payment_id|subscription_id|amount|currency|payment_status|       payment_date|payment_method|         ingest_time|
+----------+---------------+------+--------+--------------+-------------------+--------------+--------------------+
|         1|         177985| 10.40|     USD|       success|2024-06-05 00:00:00|        paypal|2026-04-22 11:16:...|
|         5|         238103|  0.00|     EUR|       success|2024-06-16 00:00:00|   credit_card|2026-04-22 11:16:...|
+----------+---------------+------+--------+--------------+-------------------+--------------+--------------------+
only showing top 2 rows



## Normalize and validate categorical values
For payment_status, currency, payment_method.<br>
In details:
+ Normalize values to a standard format
+ Validate against allowed business-domain values
+ Quarantine or flag invalid rows

Observed distinct values are:
+ payment_status: refunded, success, failed
+ currency: GBP, EUR, USD, VND
+ payment_method: google_pay, paypal, apple_pay, credit_card, bank_transfer, invoice

In [10]:
# 1) Define allowed categorical values
valid_payment_status = ["refunded", "success", "failed"]
valid_currency = ["GBP", "EUR", "USD", "VND"]
valid_payment_method = [
    "google_pay",
    "paypal",
    "apple_pay",
    "credit_card",
    "bank_transfer",
    "invoice"
]

# 2) Normalize categorical columns
df_payments_norm = (
    df3
    .withColumn("payment_status", F.lower(F.trim(F.col("payment_status"))))
    .withColumn("currency", F.upper(F.trim(F.col("currency"))))
    .withColumn("payment_method", F.lower(F.trim(F.col("payment_method"))))
)

# 3) Validate categorical columns
df_payments_validated = (
    df_payments_norm
    .withColumn(
        "is_valid_payment_status",
        F.col("payment_status").isin(valid_payment_status)
    )
    .withColumn(
        "is_valid_currency",
        F.col("currency").isin(valid_currency)
    )
    .withColumn(
        "is_valid_payment_method",
        F.col("payment_method").isin(valid_payment_method)
    )
    .withColumn(
        "is_valid_record",
        F.col("is_valid_payment_status")
        & F.col("is_valid_currency")
        & F.col("is_valid_payment_method")
    )
)

df4 = df_payments_validated.filter(F.col("is_valid_record")).drop("is_valid_payment_status",
                                                                                 "is_valid_currency",
                                                                                 "is_valid_payment_method",
                                                                                 "is_valid_record")

df4_quarantine = (
    df_payments_validated
    .filter(~F.col("is_valid_record"))
    .withColumn(
        "validation_error",
        F.concat_ws(
            "; ",
            F.when(~F.col("is_valid_payment_status"), F.lit("Invalid payment_status")),
            F.when(~F.col("is_valid_currency"), F.lit("Invalid currency")),
            F.when(~F.col("is_valid_payment_method"), F.lit("Invalid payment_method"))
        ))
)



In [11]:
# df4.show(5)

[Stage 28:======================================>                   (2 + 1) / 3]

+----------+---------------+------+--------+--------------+-------------------+--------------+--------------------+
|payment_id|subscription_id|amount|currency|payment_status|       payment_date|payment_method|         ingest_time|
+----------+---------------+------+--------+--------------+-------------------+--------------+--------------------+
|         1|         177985| 10.40|     USD|       success|2024-06-05 00:00:00|        paypal|2026-04-22 11:16:...|
|         5|         238103|  0.00|     EUR|       success|2024-06-16 00:00:00|   credit_card|2026-04-22 11:16:...|
|         7|         212493| 13.98|     USD|       success|2025-07-27 00:00:00|   credit_card|2026-04-22 11:16:...|
|        15|          45975|  0.00|     VND|       success|2025-08-06 00:00:00|    google_pay|2026-04-22 11:16:...|
|        16|         165251| 27.68|     VND|       success|2024-05-26 00:00:00|   credit_card|2026-04-22 11:16:...|
+----------+---------------+------+--------+--------------+-------------

## Validate referential integrity for subscription_id 
Based on subscription_id in the subscription silver table

In [12]:
silver_subscriptions_path = "./silver/subscriptions"
df_silver_subscriptions = (
    spark.read
    .format("delta")
    .load(silver_subscriptions_path)
)

df_silver_subscriptions.show(5)

+---------------+-------+-------+----------+----------+---------+-----------+-------------------+
|subscription_id|user_id|plan_id|start_date|  end_date|   status|current_mrr|         created_at|
+---------------+-------+-------+----------+----------+---------+-----------+-------------------+
|             14|   6835|      3|2027-04-04|2027-08-18|cancelled|     370.16|2027-04-05 05:21:59|
|             20|    484|      1|2025-05-03|2025-11-30|cancelled|       1.31|2025-05-03 10:30:46|
|             21|  44487|     15|2023-04-11|      NULL|   active|      42.27|2023-04-11 23:55:29|
|             23|   3592|      1|2025-02-10|      NULL|   active|       1.31|2025-02-11 06:43:16|
|             24|  34704|     19|2024-06-20|2024-10-21|  expired|      13.98|2024-06-20 15:07:41|
+---------------+-------+-------+----------+----------+---------+-----------+-------------------+
only showing top 5 rows



In [13]:
# Keep only the reference key from silver subscriptions
df_silver_subscriptions_ref = (
    df_silver_subscriptions
    .select("subscription_id")
    .dropDuplicates()
)

# Invalid records: subscription_id in payments does NOT exist in silver subscriptions
df5_quarantine = (
    df4.alias("p")
    .join(
        df_silver_subscriptions_ref.alias("s"),
        on="subscription_id",
        how="left_anti"
    )
)

# Valid records: subscription_id exists in silver subscriptions
df5 = (
    df4.alias("p")
    .join(
        df_silver_subscriptions_ref.alias("s"),
        on="subscription_id",
        how="inner"
    )
)
df5.count(), df5_quarantine.count()

(316795, 33205)

## Derive currency

Derive reporting amount for amount_usd or other converted reporting currency.

Need to add the exchange rate into the dataset before hand, because each currency has a fixed exchange rate at the time of payment.

In [16]:
# Example: 1 USD = target currency
usd_to_target_rates = {
    "USD": 1.0,
    "EUR": 0.93,
    "GBP": 0.80,
    "VND": 25500.0
}

rate_expr = create_map([lit(x) for x in chain(*usd_to_target_rates.items())])

df6 = (
    df5
    .withColumn("currency_clean", upper(trim(col("currency"))))
    .withColumn("exchange_rate", rate_expr[col("currency_clean")].cast(DecimalType(18, 6)))
    .withColumn(
        "amount_in_currency",
        (col("amount").cast(DecimalType(18, 2)) * col("exchange_rate")).cast(DecimalType(18, 2))
    )
    .drop("currency_clean")
)

In [18]:
df6.select("currency").distinct().show()

[Stage 75:==============>                                           (1 + 3) / 4]

+--------+
|currency|
+--------+
|     GBP|
|     EUR|
|     USD|
|     VND|
+--------+



## Get all quarantine tables

In [19]:
quarantine_dfs = [
    df2_quarantine,
    df4_quarantine,
    df5_quarantine
]

df_quarantine_all = reduce(
    lambda df1, df2: df1.unionByName(df2, allowMissingColumns=True),
    quarantine_dfs
)
df_quarantine_all.count()

383205

In [20]:
df_quarantine_all.show(5)

[Stage 106:===========>     (2 + 1) / 3][Stage 107:===========>     (2 + 1) / 3]

+----------+---------------+------+--------+--------------+-------------------+--------------+--------------------+-----------------------+-----------------+-----------------------+---------------+----------------+
|payment_id|subscription_id|amount|currency|payment_status|       payment_date|payment_method|         ingest_time|is_valid_payment_status|is_valid_currency|is_valid_payment_method|is_valid_record|validation_error|
+----------+---------------+------+--------+--------------+-------------------+--------------+--------------------+-----------------------+-----------------+-----------------------+---------------+----------------+
|         1|         177985| 10.40|     USD|       success|2024-06-05 00:00:00|        paypal|2026-04-22 11:16:...|                   NULL|             NULL|                   NULL|           NULL|            NULL|
|         5|         238103|  0.00|     EUR|       success|2024-06-16 00:00:00|   credit_card|2026-04-22 11:16:...|                   NULL| 

In [22]:
df_silver_payments = df6

In [24]:
silver_path = "./silver/payments"

# Columns from incoming batch
silver_cols = df_silver_payments.columns

# Ensure proper timestamps
df_upsert = (
    df_silver_payments
    .select(*silver_cols)
)

# Deduplicate incoming batch
w = Window.partitionBy("payment_id").orderBy(F.col("payment_date").desc(), F.col("ingest_time").desc())

df_upsert = (
    df_upsert
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

# MERGE (Upsert)
if DeltaTable.isDeltaTable(spark, silver_path):
    print("Delta table already exists")
    delta_target = DeltaTable.forPath(spark, silver_path)

    (
        delta_target.alias("target")
        .merge(
            df_upsert.alias("source"),
            """
            target.payment_id = source.payment_id
            AND target.payment_date = source.payment_date
            """
        )
        # Update only if incoming row is newer
        .whenMatchedUpdate(
            condition="source.ingest_time > target.ingest_time",
            set={c: f"source.{c}" for c in silver_cols}
        )
        .whenNotMatchedInsert(
            values={c: f"source.{c}" for c in silver_cols}
        )
        .execute()
    )

else:
    print("Delta table does not exist yet")
    (
        df_upsert
        .write
        .format("delta")
        .mode("overwrite")
        .save(silver_path)
    )


Delta table does not exist yet


In [25]:
df_payments_silver_read = (
    spark.read
    .format("delta")
    .load(silver_path)
)

df_payments_silver_read.show(5)

+---------------+----------+------+--------+--------------+-------------------+--------------+--------------------+-------------+------------------+
|subscription_id|payment_id|amount|currency|payment_status|       payment_date|payment_method|         ingest_time|exchange_rate|amount_in_currency|
+---------------+----------+------+--------+--------------+-------------------+--------------+--------------------+-------------+------------------+
|           5872|         3| 27.68|     USD|       success|2024-11-03 00:00:00|   credit_card|2026-04-22 11:16:...|     1.000000|             27.68|
|         130522|        14| 43.83|     EUR|       success|2025-01-24 00:00:00| bank_transfer|2026-04-22 11:16:...|     0.930000|             40.76|
|         197620|        32| 10.40|     USD|       success|2023-09-14 00:00:00|        paypal|2026-04-22 11:16:...|     1.000000|             10.40|
|         249767|        35| 23.62|     USD|       success|2024-03-06 00:00:00|   credit_card|2026-04-22 1

In [26]:
spark.stop()